In [ ]:
!pip install vaderSentiment plotly anthropic google-api-python-client -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 12.6 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata
userdata.get('YOUTUBE_API_KEY')
SKC_CHANNEL_ID = "UCDcDw3eCZvwNQdLfSUzhkTg"  # Sporting KC official channel

In [ ]:
from googleapiclient.discovery import build
import pandas as pd

youtube = build("youtube", "v3", developerKey=YOUTUBE_API_KEY)

# Get SKC's most recent videos
videos = youtube.search().list(
    channelId=SKC_CHANNEL_ID,
    part="snippet",
    type="video",
    order="date",
    maxResults=10
).execute()

video_ids = [v["id"]["videoId"] for v in videos["items"]]
video_titles = [v["snippet"]["title"] for v in videos["items"]]

for vid, title in zip(video_ids, video_titles):
    print(vid, "|", title)

print("Videos found:", len(videos["items"]))
print(videos)

Ai0WaNhJMY4 | &quot;This is dinosaur stuff!&quot; | Gen Z Players Try 90s Toys
2f9wFodpm6Q | Post-Game Press Conference | SKC 1 – 4 COL | March 21, 2026
UuWoGyEUs3k | Shapi scores a precision goal against Colorado
50pM7YLNsBk | Pre-Game Press Conference | Rapha Wicky &amp; Dejan Joveljić | SKC vs. COL
2MjOfbF6Vwc | Do they know this sound? 🔊
Qc1PVrbRrJI | Jansen Miller Prepping for FIFA World Cup 2026™ | Off the Pitch | Ep. 2
tFRYzExVbC4 | Inside Sporting KC&#39;s Win at LA Galaxy | Behind The Scenes + Locker Room Speech
tDcfuAPe28k | Welcome to KC, Diego Borges
H9k8yo7WDHw | Game Balls for Berg Johnsen, Wicky &amp; Lee After LA Win
O_RVWgnvtfw | A perfect debut ✨
Videos found: 10
{'kind': 'youtube#searchListResponse', 'etag': '9tNf9l_ns3Mw5LrCZL-ZF5y8aek', 'nextPageToken': 'CAoQAA', 'regionCode': 'US', 'pageInfo': {'totalResults': 416, 'resultsPerPage': 10}, 'items': [{'kind': 'youtube#searchResult', 'etag': 'aTbtJU8WYvpnmLr8NelYomoDOcc', 'id': {'kind': 'youtube#video', 'videoId': 'Ai

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import pandas as pd

# Best videos for fan sentiment - match reactions + highlights
TARGET_VIDEOS = {
    "2f9wFodpm6Q": "Post-Game Press Conference | SKC 1-4 COL",
    "UuWoGyEUs3k": "Shapi scores precision goal vs Colorado",
    "tFRYzExVbC4": "Inside SKC Win at LA Galaxy",
    "SWx7P9ep4As": "Post-Game Press Conference | SKC 0-1 SD",
    "KJpzO-xfxis": "Sporting Park erupts after Pulskamp penalty save",
    "FcKqhG_d94c": "Joveljic scores his second of the night",
}

def get_comments(video_id, max_comments=100):
    comments = []
    try:
        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=100,
            textFormat="plainText",
            order="relevance"
        )
        response = request.execute()
        for item in response["items"]:
            c = item["snippet"]["topLevelComment"]["snippet"]
            comments.append({
                "video_id": video_id,
                "text": c["textDisplay"],
                "likes": c["likeCount"],
                "published": c["publishedAt"]
            })
    except Exception as e:
        print(f"Skipping {video_id}: {e}")
    return comments

# Pull comments from all target videos
all_comments = []
for vid_id, title in TARGET_VIDEOS.items():
    print(f"Pulling: {title}")
    comments = get_comments(vid_id)
    for c in comments:
        c["video_title"] = title
    all_comments.extend(comments)
    print(f"  → {len(comments)} comments")

df = pd.DataFrame(all_comments)
print(f"\nTotal comments collected: {len(df)}")
df.head()

Pulling: Post-Game Press Conference | SKC 1-4 COL
  → 0 comments
Pulling: Shapi scores precision goal vs Colorado
  → 3 comments
Pulling: Inside SKC Win at LA Galaxy
  → 5 comments
Pulling: Post-Game Press Conference | SKC 0-1 SD
  → 4 comments
Pulling: Sporting Park erupts after Pulskamp penalty save
  → 5 comments
Pulling: Joveljic scores his second of the night
  → 5 comments

Total comments collected: 22


,video_id,text,likes,published,video_title
0,UuWoGyEUs3k,Sporting it’s ok we lost we can win next time ...,2,2026-03-22T03:38:44Z,Shapi scores precision goal vs Colorado
1,UuWoGyEUs3k,skc will never win without an experienced GK,1,2026-03-22T05:31:35Z,Shapi scores precision goal vs Colorado
2,UuWoGyEUs3k,🇷🇺🇷🇺🇷🇺,1,2026-03-22T17:42:45Z,Shapi scores precision goal vs Colorado
3,tFRYzExVbC4,Honestly super nice to see Lee travel with the...,3,2026-03-18T00:04:43Z,Inside SKC Win at LA Galaxy
4,tFRYzExVbC4,What a gritty/resilient group! Love it.,1,2026-03-18T11:05:52Z,Inside SKC Win at LA Galaxy


22 comments is thin but workable — YouTube just doesn't have many comments on these videos yet since they're very recent. Let's widen the net by pulling from more videos.

In [ ]:
# Pull from ALL 25 videos we found earlier for maximum data
ALL_VIDEO_IDS = [
    "Ai0WaNhJMY4", "2f9wFodpm6Q", "UuWoGyEUs3k", "50pM7YLNsBk",
    "2MjOfbF6Vwc", "Qc1PVrbRrJI", "tFRYzExVbC4", "tDcfuAPe28k",
    "H9k8yo7WDHw", "O_RVWgnvtfw", "dx8B9G09wH4", "FbJkFAatPjc",
    "kGLD8aagOms", "SWx7P9ep4As", "g8ZmxzNkPjo", "FNOXoCi2gMA",
    "KJpzO-xfxis", "rBZQNvmlaGk", "FcKqhG_d94c", "byZhZ3ZwyII",
    "RKAJT4uRyOo", "re1y1RQ4cw8", "c7irTKudovw", "ojpCvMgIjKw",
    "8kiqw6MS2jA"
]

ALL_VIDEO_TITLES = [
    "Gen Z Players Try 90s Toys", "Post-Game Press Conference SKC 1-4 COL",
    "Shapi scores precision goal vs Colorado", "Pre-Game Press Conference SKC vs COL",
    "Do they know this sound", "Jansen Miller Prepping for FIFA World Cup",
    "Inside SKC Win at LA Galaxy", "Welcome to KC Diego Borges",
    "Game Balls for Berg Johnsen Wicky Lee", "A perfect debut",
    "Lasse scores game winner MLS debut", "Focused for Saturdays Showdown",
    "Would you rather be a baby goat", "Post-Game Press Conference SKC 0-1 SD",
    "Welcome to Off the Pitch Ep1", "Largest animal you could fight off",
    "Sporting Park erupts Pulskamp penalty save", "Pulskamp denies Columbus penalty",
    "Joveljic scores second of the night", "Joveljic ties it up vs Columbus",
    "Kansas City here we come", "How does it feel to be back at Sporting Park",
    "Introducing Taylor Calheira Ethan Bartlow", "SKC at San Jose Post-Game",
    "Pulskamp Wicky Lee Press Conference"
]

all_comments = []
for vid_id, title in zip(ALL_VIDEO_IDS, ALL_VIDEO_TITLES):
    comments = get_comments(vid_id)
    for c in comments:
        c["video_title"] = title
    all_comments.extend(comments)

df = pd.DataFrame(all_comments)
print(f"Total comments: {len(df)}")

# Run VADER sentiment on everything
sia = SentimentIntensityAnalyzer()
df["compound"] = df["text"].apply(lambda x: sia.polarity_scores(str(x))["compound"])
df["sentiment"] = df["compound"].apply(
    lambda c: "positive" if c >= 0.05 else ("negative" if c <= -0.05 else "neutral")
)

print(df["sentiment"].value_counts())
print(f"\nAvg sentiment score: {df['compound'].mean():.3f}")
df[["text","sentiment","compound","video_title"]].head(10)

Total comments: 72
sentiment
positive    44
neutral     16
negative    12
Name: count, dtype: int64

Avg sentiment score: 0.364


,text,sentiment,compound,video_title
0,"Hello, it's nice that someone finally has some...",positive,0.8126,Gen Z Players Try 90s Toys
1,I swear i am not old... but dang.. 😂,positive,0.5789,Gen Z Players Try 90s Toys
2,Sporting it’s ok we lost we can win next time ...,positive,0.5719,Shapi scores precision goal vs Colorado
3,skc will never win without an experienced GK,negative,-0.4717,Shapi scores precision goal vs Colorado
4,🇷🇺🇷🇺🇷🇺,neutral,0.0000,Shapi scores precision goal vs Colorado
5,"""Sporting KC has been a big club"" \n\namen Raf...",positive,0.5859,Pre-Game Press Conference SKC vs COL
6,Good luck on Saturday boys!,positive,0.7345,Do they know this sound
7,Good luck on Saturday boys,positive,0.7096,Do they know this sound
8,You guys should ask Messi or Ronaldo 👀,neutral,0.0000,Do they know this sound
9,Honestly super nice to see Lee travel with the...,positive,0.8658,Inside SKC Win at LA Galaxy


In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── Color scheme ──────────────────────────────────────────
POS_COLOR  = "#1a7a4a"
NEG_COLOR  = "#c0392b"
NEU_COLOR  = "#7f8c8d"
SKC_BLUE   = "#0e1e5b"
SKC_LIGHT  = "#e8eaf6"

# ── 1. Sentiment breakdown donut ──────────────────────────
counts = df["sentiment"].value_counts()

fig1 = go.Figure(go.Pie(
    labels=counts.index.tolist(),
    values=counts.values.tolist(),
    hole=0.55,
    marker_colors=[POS_COLOR, NEU_COLOR, NEG_COLOR],
    textinfo="label+percent",
    textfont_size=13
))
fig1.update_layout(
    title="Overall Fan Sentiment — SKC YouTube (March 2026)",
    title_font=dict(size=16, color=SKC_BLUE),
    showlegend=False,
    annotations=[dict(text=f"{len(df)}<br>comments", x=0.5, y=0.5,
                      font_size=14, showarrow=False)]
)
fig1.show()

# ── 2. Sentiment by video ─────────────────────────────────
video_sent = df.groupby(["video_title","sentiment"]).size().unstack(fill_value=0)
for col in ["positive","neutral","negative"]:
    if col not in video_sent.columns:
        video_sent[col] = 0

# Shorten titles for display
video_sent.index = [t[:45] + "..." if len(t) > 45 else t for t in video_sent.index]

fig2 = go.Figure()
fig2.add_trace(go.Bar(name="Positive", x=video_sent.index,
                      y=video_sent["positive"], marker_color=POS_COLOR))
fig2.add_trace(go.Bar(name="Neutral",  x=video_sent.index,
                      y=video_sent["neutral"],  marker_color=NEU_COLOR))
fig2.add_trace(go.Bar(name="Negative", x=video_sent.index,
                      y=video_sent["negative"], marker_color=NEG_COLOR))
fig2.update_layout(
    barmode="stack",
    title="Sentiment by Video",
    title_font=dict(size=16, color=SKC_BLUE),
    xaxis_tickangle=-35,
    xaxis_title="",
    yaxis_title="Comment count",
    legend=dict(orientation="h", y=1.08),
    height=480
)
fig2.show()

# ── 3. Compound score scatter ─────────────────────────────
df["short_title"] = df["video_title"].apply(lambda t: t[:40]+"..." if len(t)>40 else t)
df["color"] = df["sentiment"].map({"positive": POS_COLOR,
                                    "neutral":  NEU_COLOR,
                                    "negative": NEG_COLOR})

fig3 = px.strip(
    df, x="short_title", y="compound", color="sentiment",
    color_discrete_map={"positive": POS_COLOR,
                        "neutral":  NEU_COLOR,
                        "negative": NEG_COLOR},
    hover_data={"text": True, "likes": True, "compound": ":.3f"},
    title="Individual Comment Scores by Video"
)
fig3.add_hline(y=0.05,  line_dash="dot", line_color=POS_COLOR, opacity=0.4)
fig3.add_hline(y=-0.05, line_dash="dot", line_color=NEG_COLOR, opacity=0.4)
fig3.update_layout(
    xaxis_tickangle=-35,
    xaxis_title="",
    yaxis_title="VADER compound score (−1 to +1)",
    height=460,
    title_font=dict(size=16, color=SKC_BLUE)
)
fig3.show()

# ── 4. Top comments table ─────────────────────────────────
top_pos = df[df["sentiment"]=="positive"].nlargest(5,"compound")[["text","compound","video_title"]]
top_neg = df[df["sentiment"]=="negative"].nsmallest(5,"compound")[["text","compound","video_title"]]

print("=" * 60)
print("TOP 5 MOST POSITIVE COMMENTS")
print("=" * 60)
for _, row in top_pos.iterrows():
    print(f"[{row['compound']:.2f}] {row['text'][:80]}")
    print(f"       → {row['video_title']}\n")

print("=" * 60)
print("TOP 5 MOST NEGATIVE COMMENTS")
print("=" * 60)
for _, row in top_neg.iterrows():
    print(f"[{row['compound']:.2f}] {row['text'][:80]}")
    print(f"       → {row['video_title']}\n")

# ── 5. Export CSV ─────────────────────────────────────────
df.to_csv("skc_fan_sentiment.csv", index=False)
print("\n✓ Saved: skc_fan_sentiment.csv")
print(f"✓ {len(df)} comments | Avg score: {df['compound'].mean():.3f}")

TOP 5 MOST POSITIVE COMMENTS
[0.98] I love this team I have been supporting for a while and watching the win was abs
       → Inside SKC Win at LA Galaxy

[0.98] LETS GO SKC❤❤❤❤❤❤
       → Kansas City here we come

[0.96] Let’s go ❤❤❤❤
       → Focused for Saturdays Showdown

[0.93] Loving what im hearing from coach wicky. The rebuild will take time and the wins
       → Post-Game Press Conference SKC 0-1 SD

[0.90] Let’s go boys keep it up❤❤🎉
       → Sporting Park erupts Pulskamp penalty save

TOP 5 MOST NEGATIVE COMMENTS
[-0.67] Crazy fake
       → Joveljic ties it up vs Columbus

[-0.64] Fight off LAFC . Fight off Portland Timbers . 🙏
       → Largest animal you could fight off

[-0.51] He looks like he gonna turn in into prime Sergio Ramos 😱
       → Welcome to KC Diego Borges

[-0.51] Need to practice corners and set pieces. Those are missed opportunities for goal
       → Post-Game Press Conference SKC 0-1 SD

[-0.48] Iam the wessam abou ali soory the for clmbus crew😢
       → P

In [ ]:
import json

# Prepare all data for the dashboard
dashboard_data = {
    "summary": {
        "total": int(len(df)),
        "positive": int((df['sentiment']=='positive').sum()),
        "negative": int((df['sentiment']=='negative').sum()),
        "neutral": int((df['sentiment']=='neutral').sum()),
        "avg_score": round(float(df['compound'].mean()), 3)
    },
    "by_video": df.groupby('video_title').apply(lambda x: {
        "positive": int((x['sentiment']=='positive').sum()),
        "negative": int((x['sentiment']=='negative').sum()),
        "neutral": int((x['sentiment']=='neutral').sum()),
        "avg": round(float(x['compound'].mean()), 3)
    }).to_dict(),
    "comments": df[['text','sentiment','compound','video_title','likes']].to_dict(orient='records')
}

# Clean up likes column (convert to int)
for c in dashboard_data['comments']:
    c['compound'] = round(float(c['compound']), 4)
    c['likes'] = int(c['likes'])

print(json.dumps(dashboard_data['summary'], indent=2))
print(f"\nVideos: {len(dashboard_data['by_video'])}")
print(f"Comments: {len(dashboard_data['comments'])}")

# Save JSON
with open('skc_data.json', 'w') as f:
    json.dump(dashboard_data, f)
print("\n✓ skc_data.json saved")

{
  "total": 72,
  "positive": 44,
  "negative": 12,
  "neutral": 16,
  "avg_score": 0.364
}

Videos: 22
Comments: 72

✓ skc_data.json saved


/tmp/ipykernel_11287/1529335615.py:12: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

